## Ai Software development Agent System

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_core.tools import tool
from pathlib import Path
import subprocess
import os

In [3]:
# ============================================================
# PROJECT WORKSPACE
# ============================================================

PROJECT_ROOT = Path("generated_projects").resolve()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


In [4]:
def safe_path(path: str) -> Path:
    """
    Convert a relative project path into a safe absolute path.

    This prevents the agent from accessing files outside
    the generated project workspace.
    """

    target = (PROJECT_ROOT / path).resolve()

    if not str(target).startswith(str(PROJECT_ROOT)):
        raise ValueError("Access denied: path is outside project workspace.")

    return target

In [6]:
@tool
def create_file(path: str, content: str) -> str:
    """
    Create a new file inside the project workspace.

    Args:
        path: Relative path of the file.
        content: Content to write into the file.

    Example:
        create_file(
            "app/main.py",
            "print('Hello World')"
        )
    """

    try:

        file_path = safe_path(path)

        # Create parent directories
        file_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        # Prevent accidental overwrite
        if file_path.exists():
            return f"File already exists: {path}"

        file_path.write_text(
            content,
            encoding="utf-8"
        )

        return f"File created successfully: {path}"

    except Exception as e:
        return f"Error creating file: {str(e)}"



In [7]:
path = 'test/main.py'
result = create_file.invoke({
    "path": path,
    "content": "print('Hello from main.py')"
})

print(result)

File already exists: test/main.py


In [10]:
path2 = 'test/abcd.py'
result = create_file.invoke({
    "path": path2,
    "content": "print('Another file testing')"
})

print(result)

File already exists: test/abcd.py


In [11]:
file_path = safe_path(path)
file_path

WindowsPath('F:/Artificial_Intellgence/Agentic_Ai_Course/Agentic_Ai_Detail_Course/Code_Lectures/Phase_03_Langchain_Agent_Fundamental/Project/Ai-Software-Development-Agent-System/research/generated_projects/test/abcd.py')

In [12]:
# ============================================================
# 2. READ FILE
# ============================================================

@tool
def read_file(path: str) -> str:
    """
    Read the contents of a project file.
    """

    try:

        file_path = safe_path(path)

        if not file_path.exists():
            return f"File does not exist: {path}"

        if not file_path.is_file():
            return f"Path is not a file: {path}"

        content = file_path.read_text(
            encoding="utf-8"
        )

        return content

    except Exception as e:
        return f"Error reading file: {str(e)}"


In [13]:
read = read_file.invoke({
    "path":"F:\\Artificial_Intellgence\\Agentic_Ai_Course\\Agentic_Ai_Detail_Course\\Code_Lectures\\Phase_03_Langchain_Agent_Fundamental\\Project\\Ai-Software-Development-Agent-System\\research\\generated_projects\\test\\main.py"
})

read

'This is update agent'

In [14]:
# ============================================================
# 3. UPDATE FILE
# ============================================================

@tool
def update_file(path: str, content: str) -> str:
    """
    Replace the entire contents of an existing file.

    The Developer Agent can use this tool when fixing code.
    """

    try:

        file_path = safe_path(path)

        if not file_path.exists():
            return f"File does not exist: {path}"

        file_path.write_text(
            content,
            encoding="utf-8"
        )

        return f"File updated successfully: {path}"

    except Exception as e:
        return f"Error updating file: {str(e)}"



In [15]:
update = update_file.invoke({
    "path":path,
    "content":"print('Agentic ai')"
})
update

'File updated successfully: test/abcd.py'

In [16]:
# ============================================================
# 4. DELETE FILE
# ============================================================

@tool
def delete_file(path: str) -> str:
    """
    Delete a file from the project workspace.
    """

    try:

        file_path = safe_path(path)

        if not file_path.exists():
            return f"File does not exist: {path}"

        if not file_path.is_file():
            return f"Path is not a file: {path}"

        file_path.unlink()

        return f"File deleted successfully: {path}"

    except Exception as e:
        return f"Error deleting file: {str(e)}"


In [17]:
delete = delete_file.invoke({
    "path":path
})

In [18]:
# ============================================================
# 5. LIST PROJECT FILES
# ============================================================

@tool
def list_files(directory: str = "") -> str:
    """
    List files and directories inside the project workspace.

    Example:
        list_files("")
        list_files("app")
    """

    try:

        directory_path = safe_path(directory)

        if not directory_path.exists():
            return f"Directory does not exist: {directory}"

        items = []

        for item in directory_path.rglob("*"):

            relative_path = item.relative_to(PROJECT_ROOT)

            if item.is_dir():
                items.append(f"[DIR]  {relative_path}")
            else:
                items.append(f"[FILE] {relative_path}")

        if not items:
            return "Directory is empty."

        return "\n".join(items)

    except Exception as e:
        return f"Error listing files: {str(e)}"


In [20]:
list_dir = list_files.invoke({
    "directory":"F:\\Artificial_Intellgence\\Agentic_Ai_Course\\Agentic_Ai_Detail_Course\\Code_Lectures\\Phase_03_Langchain_Agent_Fundamental\\Project\\Ai-Software-Development-Agent-System\\research\\generated_projects"
})
list_dir

'[DIR]  test\n[FILE] test\\main.py'

In [21]:
# ============================================================
# 6. RUN PYTHON FILE
# ============================================================

@tool
def run_python(path: str) -> str:
    """
    Run a Python file inside the project workspace.

    Example:
        run_python("app/main.py")
    """

    try:

        file_path = safe_path(path)

        if not file_path.exists():
            return f"File does not exist: {path}"

        if file_path.suffix != ".py":
            return "Only Python files can be executed."

        result = subprocess.run(
            ["python", str(file_path)],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            timeout=30
        )

        output = ""

        if result.stdout:
            output += f"STDOUT:\n{result.stdout}\n"

        if result.stderr:
            output += f"STDERR:\n{result.stderr}\n"

        output += f"\nReturn code: {result.returncode}"

        return output

    except subprocess.TimeoutExpired:
        return "Execution stopped: program exceeded 30 seconds."

    except Exception as e:
        return f"Error running Python file: {str(e)}"


In [24]:
run_file = run_python.invoke({
    "path":"F:\\Artificial_Intellgence\\Agentic_Ai_Course\\Agentic_Ai_Detail_Course\\Code_Lectures\\Phase_03_Langchain_Agent_Fundamental\\Project\\Ai-Software-Development-Agent-System\\research\\generated_projects\\test\\main.py"
})
run_file

'STDOUT:\nHello from main.py\n\n\nReturn code: 0'

In [25]:
# ============================================================
# 7. RUN PYTEST
# ============================================================

@tool
def run_tests() -> str:
    """
    Run pytest on the generated project.
    """

    try:

        result = subprocess.run(
            ["pytest", "-v"],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            timeout=120
        )

        output = ""

        if result.stdout:
            output += result.stdout

        if result.stderr:
            output += "\nERRORS:\n"
            output += result.stderr

        output += f"\n\nReturn code: {result.returncode}"

        if result.returncode == 0:
            output += "\n\nTEST RESULT: PASSED"
        else:
            output += "\n\nTEST RESULT: FAILED"

        return output

    except subprocess.TimeoutExpired:
        return "Testing stopped: pytest exceeded 120 seconds."

    except Exception as e:
        return f"Error running tests: {str(e)}"


In [31]:
result = run_tests.invoke({})
print(result)

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- G:\InstallSoftWare\Anaconda_install\python.exe
cachedir: .pytest_cache
rootdir: F:\Artificial_Intellgence\Agentic_Ai_Course\Agentic_Ai_Detail_Course\Code_Lectures\Phase_03_Langchain_Agent_Fundamental\Project\Ai-Software-Development-Agent-System
configfile: pyproject.toml
plugins: anyio-4.14.2, langsmith-0.11.1
collecting ... collected 0 items

============================ no tests ran in 0.14s ============================


Return code: 5

TEST RESULT: FAILED


In [26]:
#  ============================================================
# 8. RUN RUFF
# ============================================================

@tool
def run_ruff() -> str:
    """
    Run Ruff code quality checks on the generated project.
    """

    try:

        result = subprocess.run(
            ["ruff", "check", "."],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            timeout=60
        )

        output = ""

        if result.stdout:
            output += result.stdout

        if result.stderr:
            output += "\nERRORS:\n"
            output += result.stderr

        output += f"\n\nReturn code: {result.returncode}"

        if result.returncode == 0:
            output += "\n\nCODE REVIEW: PASSED"
        else:
            output += "\n\nCODE REVIEW: ISSUES FOUND"

        return output

    except Exception as e:
        return f"Error running Ruff: {str(e)}"

In [30]:
result = run_ruff.invoke({})
print(result)

All checks passed!


Return code: 0

CODE REVIEW: PASSED


In [32]:
# ============================================================
# 9. GIT STATUS
# ============================================================

@tool
def git_status() -> str:
    """
    Show the current Git status of the generated project.
    """

    try:

        result = subprocess.run(
            ["git", "status", "--short"],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True
        )

        return result.stdout or "Working tree is clean."

    except Exception as e:
        return f"Git error: {str(e)}"


In [34]:
result = git_status.invoke({})
result

' M ../../README.md\n?? ../../.python-version\n?? ../../app.py\n?? ../../main.py\n?? ../../pyproject.toml\n?? ../../requirements.txt\n?? ../\n?? ../../src/\n?? ../../uv.lock\n'

In [35]:
# ============================================================
# 10. GIT DIFF
# ============================================================

@tool
def git_diff() -> str:
    """
    Show changes made to the generated project.
    """

    try:

        result = subprocess.run(
            ["git", "diff"],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True
        )

        return result.stdout or "No changes detected."

    except Exception as e:
        return f"Git diff error: {str(e)}"


In [37]:
result = git_diff.invoke({})
result

'diff --git a/README.md b/README.md\nindex 93442b3..193e193 100644\n--- a/README.md\n+++ b/README.md\n@@ -1 +1,2 @@\n-# Ai-Software-Development-Agent-System\n\\ No newline at end of file\n+# Ai-Software-Development-Agent-System\n+\n'

In [38]:
# ============================================================
# 11. GIT COMMIT
# ============================================================

@tool
def git_commit(message: str) -> str:
    """
    Commit current project changes to Git.

    Example:
        git_commit("Add student API")
    """

    try:

        # Add changes
        subprocess.run(
            ["git", "add", "."],
            cwd=PROJECT_ROOT,
            check=True
        )

        # Commit
        result = subprocess.run(
            ["git", "commit", "-m", message],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True
        )

        return (
            f"Git commit result:\n"
            f"{result.stdout}\n"
            f"{result.stderr}"
        )

    except Exception as e:
        return f"Git commit error: {str(e)}"

In [40]:
result = git_commit.invoke({
    "message":"Tools are created"
})
result

'Git commit result:\n[main 9c31f55] Tools are created\n 1 file changed, 1 insertion(+)\n create mode 100644 research/generated_projects/test/main.py\n\n'